## Notebook Description

The notebook will contain 5 main components:
- Configuration
- Data collection & cleaning
- Feature creation
- Model training
- Results

Each main component may contain mutiple subcomponents depending on the complexity of the component
The novelty added in this notebook is the usage of three new features:
- Quali Position
- Historical influence of quali result to the final race position on each track
- FP2 pace

Disclaimer: This is a notebook that uses as a starting point the Austin_GP_Winner_Predictor made by Mariana Antaya


### Step 0 - Package Installation

In [ ]:
# %pip install fastf1 scikit-learn xgboost matplotlib

### Step 1 - Configuration

- Import the necessary libraries 
- Configure different necessities as cache

In [45]:
from pathlib import Path
from datetime import datetime, timezone
from collections import deque, defaultdict
import pandas as pd
import numpy as np
import time
import fastf1
from fastf1.ergast import Ergast
import sklearn
#from sklearn.preprocessing import PolynomialFeatures
import matplotlib.pyplot as plt

cache_dir = Path('data_cache')
cache_dir.mkdir(exist_ok=True)
fastf1.Cache.enable_cache(str(cache_dir))

api = Ergast(result_type='pandas', limit=1000)

YEAR = 2025
START_YEAR = 2022


AUSTIN_ID = 'americas'
GRID_START_YEAR = 2014
MAX_SLOT = 20            # starting slots beyond this are merged into the last slot

### Step 2 - Data Collection

This step will be split into two substeps:
- Step 2.1 Data Collection & Cleaning as in reference notebook
- Step 2.2 Data Collection & Cleaning for new features
- Step 2.3 Final Data Aggregation

#### Step 2.1 Data Collection & Cleaning - ref notebook

In [40]:
def season_results(year):
    csv_path = cache_dir / f"results_{year}.csv"

    page = api.get_race_results(season=year)

    features = ['season', 'round', 'raceDate', 'circuitId', 'raceName']
    chunks = []
    received = 0

    while True:
        for (_, meta), result in zip(page.description.iterrows(), page.content):
            frame = pd.DataFrame(result).copy()
            for feature in features:
                frame[feature] = meta[feature]
            chunks.append(frame)
            
        try:
            time.sleep(0.4)
            page = page.get_next_result_page()
        except ValueError as exc:
            if 'No more data' in str(exc):
                break
            raise

    if not chunks:
        return pd.DataFrame()

    frame = pd.concat(chunks, ignore_index=True).drop_duplicates(['season', 'round', 'driverId'])
    frame.to_csv(csv_path, index=False)
    return frame

schedule = api.get_race_schedule(season=YEAR)
austin = schedule.loc[schedule['circuitId'].eq('americas')]
if len(austin) != 1:
    raise ValueError("No unique Austin race in this season schedule. Check target year")

target = austin.iloc[0]
target_date = pd.Timestamp(target['raceDate']).tz_localize(None).normalize()

frames = []
for year in range(START_YEAR, YEAR):
    part = season_results(year)
    if not part.empty:
        frames.append(part)
    print(f"{year}: {len(part)} driver results loaded")

raw = pd.concat(frames, ignore_index=True)
raw['raceDate'] = pd.to_datetime(raw['raceDate']).dt.tz_localize(None)
today = pd.Timestamp(datetime.now(timezone.utc).date())

history = raw.loc[(raw['raceDate'] < target_date) & (raw['raceDate'] < today)].copy()
history['position'] = pd.to_numeric(history['position'], errors='raise')
history['points'] = pd.to_numeric(history['points'], errors='raise')
history = history.sort_values(['raceDate', 'season', 'round', 'position'])

history.head()



2022: 440 driver results loaded
2023: 440 driver results loaded
2024: 479 driver results loaded


,number,position,positionText,points,grid,laps,status,driverId,driverNumber,driverCode,...,fastestLapRank,fastestLapNumber,fastestLapTime,fastestLapAvgSpeedUnits,fastestLapAvgSpeed,season,round,raceDate,circuitId,raceName
0,16,1,1,26.0,1,57,Finished,leclerc,16,LEC,...,1.0,51.0,0 days 00:01:34.570000,kph,206.018,2022,1,2022-03-20,bahrain,Bahrain Grand Prix
1,55,2,2,18.0,3,57,Finished,sainz,55,SAI,...,3.0,52.0,0 days 00:01:35.740000,kph,203.501,2022,1,2022-03-20,bahrain,Bahrain Grand Prix
2,44,3,3,15.0,5,57,Finished,hamilton,44,HAM,...,5.0,53.0,0 days 00:01:36.228000,kph,202.469,2022,1,2022-03-20,bahrain,Bahrain Grand Prix
3,63,4,4,12.0,9,57,Finished,russell,63,RUS,...,6.0,56.0,0 days 00:01:36.302000,kph,202.313,2022,1,2022-03-20,bahrain,Bahrain Grand Prix
4,20,5,5,10.0,7,57,Finished,kevin_magnussen,20,MAG,...,8.0,53.0,0 days 00:01:36.623000,kph,201.641,2022,1,2022-03-20,bahrain,Bahrain Grand Prix


### Step 2.3 Features creation

In [49]:
FEATURES = ['driver_finish_5', 'driver_points_5', 'driver_win_rate_5',
            'driver_podium_rate_5', 'team_points_5', 'team_win_rate_5',
            'circuit_finish_3', 'circuit_win_rate_3', 'driver_history_count']

drivers = defaultdict(lambda: deque(maxlen=5))
teams = defaultdict(lambda: deque(maxlen=5))
circuits = defaultdict(lambda: deque(maxlen=3))

def avg(records, column):
    return float(np.mean([r[column] for r in records])) if records else np.nan

def features(driver, team, circuit):
    d, t, c = drivers[driver], teams[team], circuits[circuit]
    return dict(zip(FEATURES, [avg(d, 'finish'), avg(d, 'points'), avg(d, 'win'),
                    avg(d, 'podium'), avg(t, 'points'), avg(t, 'win'), avg(c, 'finish'),
                    avg(c, 'win'), len(d)]))

rows = []

for (season, rnd), race in history.groupby(['season', 'round'], sort=True):
    for _, r in race.iterrows():
        rows.append({**features(r.driverId, r.constructorId, r.circuitId),
                     'season': int(season), 'round':int(rnd),
                     'race_id': f'{season}-{rnd}', 'raceName': r.raceName,
                     'circuitId': r.circuitId, 'driverId': r.driverId,
                     'driver':f'{r.givenName} {r.familyName}', 'won':int(r.position == 1)})
    for _, r in race.iterrows():
        record = dict(finish=float(r.position), points=float(r.points),
                      win=int(r.position == 1), podium=int(r.position <= 3))
        drivers[r.driverId].append(record)
        circuits[(r.driverId, r.circuitId)].append(record)
    for team, team_race in race.groupby('constructorId'):
        teams[team].append(dict(points=float(team_race.points.sum()),
                                win=int(team_race.position.eq(1).any())))

data = pd.DataFrame(rows)
data = data.loc[data.season > START_YEAR].copy()

austin_examples = data.loc[data.circuitId.eq('americas')]

austin_examples.head()

preview = austin_examples.loc[austin_examples.season.eq(austin_examples.season.max())].head(5)
print('Example: inputs available BEFORE the', int(preview.season.iloc[0]), 'Austin race')
display(preview[['driver','driver_finish_5','team_points_5','circuit_finish_3','won']]
        .rename(columns={'driver':'Driver','driver_finish_5':'Recent avg finish',
                         'team_points_5':'Team avg points','circuit_finish_3':'Austin avg finish',
                         'won':'Won the race?'})
        .replace({'Won the race?':{0:'No',1:'Yes'}}).round(1).reset_index(drop=True))
print('This preview shows 3 of the 9 inputs. Lower average finish is better; NaN means no history.')

Example: inputs available BEFORE the 2024 Austin race


,Driver,Recent avg finish,Team avg points,Austin avg finish,Won the race?
0,Charles Leclerc,2.8,23.8,NaN,Yes
1,Carlos Sainz,8.0,23.8,NaN,No
2,Max Verstappen,3.8,17.2,NaN,No
3,Lando Norris,2.8,35.6,NaN,No
4,Oscar Piastri,2.4,35.6,NaN,No


This preview shows 3 of the 9 inputs. Lower average finish is better; NaN means no history.


#### Step 2.2 Data Collection & Cleaning - My New Features


In [41]:
META_COLS = ['season', 'round', 'raceDate', 'circuitId', 'raceName']
KEEP_COLS = META_COLS + ['driverId', 'constructorId', 'grid', 'position',
                         'positionText', 'status', 'points']


def fetch_race_results(cache_name, refresh=False, **query):
    """Download race results for any Ergast filter (season=..., circuit=...).
    Cached in data_cache/<cache_name>.csv; pass refresh=True to re-download."""
    csv_path = cache_dir / f"{cache_name}.csv"
    if csv_path.exists() and not refresh:
        return pd.read_csv(csv_path, parse_dates=['raceDate'])

    page = api.get_race_results(**query)
    chunks = []
    while True:
        for (_, meta), result in zip(page.description.iterrows(), page.content):
            frame = pd.DataFrame(result).copy()
            for col in META_COLS:
                frame[col] = meta[col]
            frame['raceDate'] = pd.Timestamp(meta['raceDate']).tz_localize(None).normalize()
            chunks.append(frame)
        try:
            time.sleep(0.4)
            page = page.get_next_result_page()
        except ValueError as exc:
            if 'No more data' in str(exc):
                break
            raise

    if not chunks:
        return pd.DataFrame()

    frame = pd.concat(chunks, ignore_index=True)
    frame = frame.drop_duplicates(['season', 'round', 'driverId'])
    frame = frame[[c for c in KEEP_COLS if c in frame.columns]]
    frame.to_csv(csv_path, index=False)
    return frame

def clean_results(df, max_slot=MAX_SLOT):
    df = df.copy()
    df['raceDate'] = pd.to_datetime(df['raceDate'])
    for col in ['season', 'round', 'grid', 'position']:
        df[col] = pd.to_numeric(df[col], errors='coerce')

    unusable = df['position'].isna() | df['grid'].isna()
    if unusable.any():
        print(f"Dropping {unusable.sum()} rows without a usable grid/position")
    df = df.loc[~unusable].copy()
    df[['season', 'round', 'grid', 'position']] = (
        df[['season', 'round', 'grid', 'position']].astype(int))

    # Starting slot = actual grid slot (penalties already included).
    # grid == 0 means pit-lane start -> placed behind the last gridded car.
    df['pit_lane_start'] = df['grid'].eq(0)
    last_gridded = (df['grid'].where(df['grid'] > 0)
                      .groupby([df['season'], df['round']]).transform('max')
                      .fillna(max_slot))
    df['starting_slot'] = np.where(df['pit_lane_start'], last_gridded + 1, df['grid'])
    df['starting_slot'] = df['starting_slot'].clip(upper=max_slot).astype(int)

    df['is_win'] = df['position'].eq(1)
    df['is_podium'] = df['position'].le(3)
    df['is_top10'] = df['position'].le(10)
    df['positions_gained'] = df['starting_slot'] - df['position']

    return df.sort_values(['raceDate', 'round', 'position']).reset_index(drop=True)


def check_races(df):
    """Returns races with a data problem (should be an empty frame)."""
    per_race = df.groupby(['season', 'round']).agg(
        circuitId=('circuitId', 'first'),
        rows=('driverId', 'size'),
        unique_drivers=('driverId', 'nunique'),
        winners=('is_win', 'sum'),
        p1_starters=('starting_slot', lambda s: (s == 1).sum()),
    )
    return per_race[(per_race['winners'] != 1)
                    | (per_race['rows'] != per_race['unique_drivers'])
                    | (per_race['p1_starters'] != 1)]


In [42]:
seasons = []
for year in range(GRID_START_YEAR, YEAR + 1):
    part = fetch_race_results(f"race_results_{year}", season=year)
    seasons.append(part)
    print(f"{year}: {len(part)} rows")

austin_all = fetch_race_results(f"race_results_circuit_{AUSTIN_ID}", circuit=AUSTIN_ID)

results_all = (pd.concat(seasons + [austin_all], ignore_index=True)
                 .drop_duplicates(['season', 'round', 'driverId']))
results_all = clean_results(results_all)

# Only races strictly before the target race: this is what removes the leak
before_target = (results_all['raceDate'] < target_date) & (results_all['raceDate'] < today)
history_all = results_all.loc[before_target].copy()
austin_hist = history_all.loc[history_all['circuitId'] == AUSTIN_ID]

print("Austin seasons used:    ", sorted(austin_hist['season'].unique()))
print("Austin seasons excluded:", sorted(results_all.loc[~before_target & (results_all['circuitId'] == AUSTIN_ID), 'season'].unique()))
print("Problem races:\n", check_races(results_all))

def slot_table(df):
    """"Outcome statistics per starting slot, using every driver of every race."""

    table = df.groupby('starting_slot').agg(
        starts = ('driverId', 'size'),
        wins = ('is_win', 'sum'),
        podiums = ('is_podium', 'sum'),
        mean_finish = ('position', 'mean'),
        mean_gain = ('positions_gained', 'mean')
    )

    table['win_rate'] = table['wins'] / table['starts']
    table['podium_rate'] = table['podiums'] / table['starts']

    return table.round(3)

def race_summary(df):
    """One row per race: where the winner started and how strongly grid order predicted finishing order."""
    rows = []
    for (season, rnd), g in df.groupby(['season', 'round']):
        winner = g.loc[g['is_win'], 'starting_slot']
        rows.append({
            'season': season,
            'round': rnd,
            'raceDate': g['raceDate'].iloc[0],
            'circuitId': g['circuitId'].iloc[0],
            'winner_slot': winner.iloc[0] if len(winner) else np.nan,
            'grid_finish_corr': g['starting_slot'].rank().corr(g['position'].rank()),
        })
    return pd.DataFrame(rows)


def circuit_summary(races):
    return (races.groupby('circuitId')
                 .agg(n_races=('round', 'size'),
                      pole_win_rate=('winner_slot', lambda s: (s == 1).mean()),
                      mean_winner_slot=('winner_slot', 'mean'),
                      grid_finish_corr=('grid_finish_corr', 'mean'))
                 .sort_values('grid_finish_corr', ascending=False)
                 .round(3))


races_hist = race_summary(history_all)
circuits_hist = circuit_summary(races_hist)

print(circuits_hist.loc[[AUSTIN_ID]])   # Austin's circuit-level numbers
slot_table(austin_hist) 

2014: 407 rows
2015: 378 rows
2016: 462 rows
2017: 400 rows
2018: 420 rows
2019: 420 rows
2020: 340 rows
2021: 440 rows
2022: 440 rows
2023: 440 rows
2024: 479 rows
2025: 479 rows
Austin seasons used:     [np.int64(2012), np.int64(2013), np.int64(2014), np.int64(2015), np.int64(2016), np.int64(2017), np.int64(2018), np.int64(2019), np.int64(2021), np.int64(2022), np.int64(2023), np.int64(2024)]
Austin seasons excluded: [np.int64(2025), np.int64(2026)]
Problem races:
 Empty DataFrame
Columns: [circuitId, rows, unique_drivers, winners, p1_starters]
Index: []
           n_races  pole_win_rate  mean_winner_slot  grid_finish_corr
circuitId                                                            
americas        12          0.417             2.083             0.499


,starts,wins,podiums,mean_finish,mean_gain,win_rate,podium_rate
starting_slot,,,,,,,
1,12,5,9,4.833,-3.833,0.417,0.750
2,12,5,11,3.250,-1.250,0.417,0.917
3,12,0,6,6.833,-3.833,0.000,0.500
4,12,1,2,8.333,-4.333,0.083,0.167
5,12,0,3,7.583,-2.583,0.000,0.250
6,12,1,1,7.583,-1.583,0.083,0.083
7,12,0,1,10.333,-3.333,0.000,0.083
8,12,0,0,13.167,-5.167,0.000,0.000
9,12,0,0,8.583,0.417,0.000,0.000


### Step 2.3 Final Data Aggregation

### Step 3 Features Extraction and Creation

### Step 4 Model Creation

### Step 5 Results